# Лабораторная работа № 4. Линейная алгебра

**Предмет:** Компьютерный практикум по статистическому анализу данных  
**Язык:** Julia  
**Среда:** Jupyter Notebook / IJulia

В ноутбуке собраны примеры из раздела 4.2 и решения самостоятельных заданий из разделов 4.4.1–4.4.4 методички. Если Jupyter спросит, какое ядро использовать, выбери установленное ядро Julia (оно может называться, например, `Julia 1.x`).

> Перед сдачей запусти ячейки по порядку (`Shift + Enter`), просмотри результаты, сделай собственные скриншоты из Jupyter и допиши ФИО на титульном листе отчёта. Времена выполнения зависят от компьютера, поэтому для отчёта используй значения, полученные у себя.

## 0. Подготовка

In [ ]:
using LinearAlgebra
using Statistics
using Random

Random.seed!(42)  # чтобы случайные примеры были воспроизводимее

`LinearAlgebra` и `Statistics` входят в стандартную поставку Julia — обычно их не нужно отдельно устанавливать через `Pkg.add`. Дополнительный пакет `BenchmarkTools` потребуется только для измерения времени выполнения (`@btime`); его установка приведена перед соответствующими примерами.

## 1. Примеры из методички — раздел 4.2

### 1.1. Поэлементные операции над массивами (4.2.1)

In [ ]:
# Матрица 4×3 со случайными целыми числами от 1 до 20
a = rand(1:20, (4, 3))
display(a)

@show sum(a)                 # сумма всех элементов
@show sum(a, dims=1)         # суммы по столбцам
@show sum(a, dims=2)         # суммы по строкам

@show prod(a)                # произведение всех элементов
@show prod(a, dims=1)        # произведения по столбцам
@show prod(a, dims=2)        # произведения по строкам

@show mean(a)                # среднее по всем элементам
@show mean(a, dims=1)        # среднее по столбцам
@show mean(a, dims=2)        # среднее по строкам

### 1.2. Транспонирование, след, ранг, определитель и обратная матрица (4.2.2)

In [ ]:
b = rand(1:20, (4, 4))
display(b)

@show transpose(b)           # транспонированная матрица
@show tr(b)                  # след: сумма элементов главной диагонали
@show diag(b)                # диагональные элементы как вектор
@show rank(b)                # ранг
@show det(b)                 # определитель

# Обратная матрица существует только при ненулевом определителе
if !isapprox(det(b), 0.0)
    @show inv(b)
else
    println("Матрица вырождена: обратной матрицы нет.")
end

@show pinv(a)                # псевдообратная матрица для прямоугольной матрицы a

### 1.3. Нормы векторов, расстояние и угол (4.2.3)

In [ ]:
X = [2, 4, -5]
Y = [1, -1, 3]

@show norm(X)                              # евклидова норма
@show norm(X, 1)                           # 1-норма
@show norm(X - Y)                          # расстояние между векторами
@show sqrt(sum((X - Y).^2))                # проверка по определению
@show acos(dot(X, Y) / (norm(X) * norm(Y))) # угол в радианах

# Нормы матрицы и поворот на 180°
d = [5 -4 2; -1 2 3; -2 1 0]
@show opnorm(d)      # операторная 2-норма
@show opnorm(d, 1)   # операторная 1-норма
@show rot180(d)
@show reverse(d, dims=1)  # переворот строк
@show reverse(d, dims=2)  # переворот столбцов

### 1.4. Матричное умножение, единичная матрица и скалярное произведение (4.2.4)

In [ ]:
A = rand(1:10, (2, 3))
B = rand(1:10, (3, 4))
@show A
@show B
@show A * B

@show Matrix{Int}(I, 3, 3)  # единичная матрица 3×3
X = [2, 4, -5]
Y = [1, -1, 3]
@show dot(X, Y)
@show X' * Y

### 1.5. Решение СЛАУ и LU-разложение (4.2.5)

In [ ]:
A = rand(3, 3)
x_expected = ones(3)
b = A * x_expected

@show A
@show b
x_solution = A \ b
@show x_solution                # должно быть примерно [1.0, 1.0, 1.0]

Alu = lu(A)
@show Alu.L
@show Alu.U
@show Alu.p
@show A \ b
@show Alu \ b
@show det(A)
@show det(Alu)

### 1.6. QR-разложение и собственные значения

In [ ]:
Aqr = qr(A)
@show Aqr.R
Q = Matrix(Aqr.Q)
@show Q' * Q                         # должно быть примерно единичной матрицей

Asym = A + A'                       # симметричная матрица
@show issymmetric(Asym)
AsymEig = eigen(Symmetric(Asym))
@show AsymEig.values                  # собственные значения
@show AsymEig.vectors                 # собственные векторы
@show AsymEig.vectors' * Asym * AsymEig.vectors  # близко к диагональной матрице

# Восстановление исходной симметричной матрицы
A_reconstructed = AsymEig.vectors * Diagonal(AsymEig.values) * AsymEig.vectors'
@show isapprox(A_reconstructed, Asym)

### 1.7. Специальные структуры и измерение времени

Следующая часть методички использует `BenchmarkTools`. Установи пакет один раз, выполнив ячейку ниже. Для ноутбука с ограниченными ресурсами большой пример с матрицей размера 1 000 000 × 1 000 000 оставлен в отдельной необязательной ячейке.

In [ ]:
import Pkg
Pkg.add("BenchmarkTools")
using BenchmarkTools

In [ ]:
# Сравнение нахождения собственных значений
n = 1000
A_big = randn(n, n)
Asym_big = A_big + A_big'
Asym_noisy = copy(Asym_big)
Asym_noisy[1, 2] += 5eps()
Asym_explicit = Symmetric(Asym_noisy)

@show issymmetric(Asym_big)
@show issymmetric(Asym_noisy)

@btime eigvals($Asym_big) samples=3 evals=1
@btime eigvals($Asym_noisy) samples=3 evals=1
@btime eigvals($Asym_explicit) samples=3 evals=1

**Как интерпретировать сравнение:** `Asym_big` является симметричной; после изменения одного элемента `Asym_noisy` она перестаёт быть симметричной. Обёртка `Symmetric(...)` явно сообщает Julia, что матрицу нужно обрабатывать как симметричную. Конкретные времена зависят от компьютера и версии Julia.

In [ ]:
# Необязательный ресурсоёмкий пример из методички.
# Матрица не хранится как плотная 1 000 000×1 000 000: сохраняются только две диагонали.
n_tri = 1_000_000
A_tri = SymTridiagonal(randn(n_tri), randn(n_tri - 1))

# Ограничиваем количество измерений, потому что задача большая.
@btime eigmax($A_tri) samples=3 evals=1

### 1.8. Рациональные числа

In [ ]:
Arational = Rational{BigInt}.(rand(1:10, 3, 3)) ./ 10
x_expected = fill(1, 3)
b_rational = Arational * x_expected

@show Arational
@show b_rational
@show Arational \ b_rational
@show lu(Arational)

## 2. Самостоятельные задания — раздел 4.4

### 2.1. Произведение векторов (4.4.1)

In [ ]:
v = [1, 2, 3]

dot_v = dot(v, v)  # скалярное произведение вектора на себя
outer_v = v * v'  # внешнее произведение: матрица 3×3

@show v
@show dot_v
@show outer_v

Для выбранного вектора `v = [1, 2, 3]` скалярное произведение равно **14**. Внешнее произведение — матрица, у которой элемент в позиции `(i, j)` равен `v[i] * v[j]`.

### 2.2. Системы линейных уравнений (4.4.2)

Функция ниже сравнивает ранг матрицы коэффициентов и ранг расширенной матрицы. Так мы отличаем несовместные системы от систем с решениями. Если решение единственное, функция дополнительно выводит его.

In [ ]:
function inspect_system(A, b; label="Система")
    A_float = Float64.(A)
    b_float = Float64.(b)
    rank_A = rank(A_float)
    rank_aug = rank(Float64.(hcat(A, b)))

    println("\n", label)
    println("Ранг матрицы коэффициентов: ", rank_A)
    println("Ранг расширенной матрицы: ", rank_aug)

    if rank_A < rank_aug
        println("Результат: решений нет (система несовместна).")
    elseif rank_A == size(A, 2)
        println("Результат: решение единственное.")
        println("Решение (x, y, ...): ", A_float \ b_float)
    else
        println("Результат: бесконечно много решений.")
    end
end

#### Системы с двумя неизвестными

In [ ]:
# a) x + y = 2; x - y = 3
inspect_system([1 1; 1 -1], [2, 3]; label="4.4.2.1(a)")

# b) x + y = 2; 2x + 2y = 4
inspect_system([1 1; 2 2], [2, 4]; label="4.4.2.1(b)")

# c) x + y = 2; 2x + 2y = 5
inspect_system([1 1; 2 2], [2, 5]; label="4.4.2.1(c)")

# d) x + y = 1; 2x + 2y = 2; 3x + 3y = 3
inspect_system([1 1; 2 2; 3 3], [1, 2, 3]; label="4.4.2.1(d)")

# e) x + y = 2; 2x + y = 1; x - y = 3
inspect_system([1 1; 2 1; 1 -1], [2, 1, 3]; label="4.4.2.1(e)")

# f) x + y = 2; 2x + y = 1; 3x + 2y = 3
inspect_system([1 1; 2 1; 3 2], [2, 1, 3]; label="4.4.2.1(f)")

#### Системы с тремя неизвестными

In [ ]:
# a) Два уравнения и три неизвестных
inspect_system([1 1 1; 1 -1 -2], [2, 3]; label="4.4.2.2(a)")

# b)
inspect_system([1 1 1; 2 2 -3; 3 1 1], [2, 4, 1]; label="4.4.2.2(b)")

# c)
inspect_system([1 1 1; 1 1 2; 2 2 3], [1, 0, 1]; label="4.4.2.2(c)")

# d)
inspect_system([1 1 1; 1 1 2; 2 2 3], [1, 0, 0]; label="4.4.2.2(d)")

**Проверка для отчёта:** среди систем с двумя неизвестными пункты (a) и (f) имеют единственное решение, (b) и (d) — бесконечно много решений, (c) и (e) несовместны. Среди систем с тремя неизвестными пункт (a) имеет бесконечно много решений, (b) — единственное решение, (c) — бесконечно много решений, (d) несовместен.

### 2.3. Операции с матрицами (4.4.3)

#### 2.3.1. Диагонализация матриц

Матрицы в задании симметричные. Для них используем `eigen(Symmetric(A))`. Если `V` — матрица собственных векторов, а `D` — диагональная матрица собственных значений, то выполняется `V' * A * V ≈ D` и `V * D * V' ≈ A`.

In [ ]:
function show_diagonalization(A, label)
    A_float = Float64.(A)
    F = eigen(Symmetric(A_float))
    V = F.vectors
    D = Diagonal(F.values)

    println("\n", label)
    @show F.values
    println("Диагональная матрица D:")
    display(Matrix(D))
    println("Матрица собственных векторов V:")
    display(V)
    println("Проверка V' * A * V ≈ D: ", isapprox(V' * A_float * V, Matrix(D)))
    println("Проверка V * D * V' ≈ A: ", isapprox(V * D * V', A_float))
end

show_diagonalization([1 -2; -2 1], "4.4.3.1(a)")
show_diagonalization([1 -2; -2 3], "4.4.3.1(b)")
show_diagonalization([1 -2 0; -2 1 2; 0 2 0], "4.4.3.1(c)")

#### 2.3.2. Степень матрицы и матричные корни

In [ ]:
# a) Десятая степень матрицы
A_power = [1 -2; -2 1]
A10 = A_power^10
@show A10

# b) Квадратный корень матрицы [5 -2; -2 5].
# Собственные значения положительны, поэтому действительный корень существует.
B = [5.0 -2.0; -2.0 5.0]
FB = eigen(Symmetric(B))
B_sqrt = FB.vectors * Diagonal(sqrt.(FB.values)) * FB.vectors'
@show B_sqrt
@show isapprox(B_sqrt * B_sqrt, B)

# c) Кубический корень матрицы [1 -2; -2 1].
# cbrt даёт действительный кубический корень и для отрицательных чисел.
C = [1.0 -2.0; -2.0 1.0]
FC = eigen(Symmetric(C))
C_cuberoot = FC.vectors * Diagonal(cbrt.(FC.values)) * FC.vectors'
@show C_cuberoot
@show isapprox(C_cuberoot^3, C)

# d) Квадратный корень матрицы [1 2; 2 3].
# У матрицы есть отрицательное собственное значение, поэтому используем комплексные числа.
Dmat = [1.0 2.0; 2.0 3.0]
FD = eigen(ComplexF64.(Dmat))
D_sqrt_complex = FD.vectors * Diagonal(sqrt.(FD.values)) * inv(FD.vectors)
@show D_sqrt_complex
@show isapprox(D_sqrt_complex * D_sqrt_complex, ComplexF64.(Dmat))

#### 2.3.3. Собственные значения заданной матрицы 5×5 и время выполнения

В этой матрице собственные значения вещественные, поскольку матрица симметрична.

In [ ]:
A5 = [140 97 74 168 131;
      97 106 89 131 36;
      74 89 152 144 71;
      168 131 144 54 142;
      131 36 71 142 36]

lambda = eigvals(Symmetric(Float64.(A5)))
D5 = Diagonal(lambda)        # диагональная матрица собственных значений
L5 = tril(A5)                # нижнетреугольная матрица с нулями выше диагонали

@show lambda
display(Matrix(D5))
display(L5)

println("Время вычисления собственных значений:")
@btime eigvals(Symmetric($A5)) samples=10 evals=1
println("Время формирования диагональной матрицы:")
@btime Diagonal(eigvals(Symmetric($A5))) samples=10 evals=1
println("Время формирования нижнетреугольной матрицы:")
@btime tril($A5) samples=10 evals=1

Ожидаемые собственные значения (с точностью до округления): приблизительно `[-128.4932, -55.8878, 42.7522, 87.1611, 542.4677]`. В отчёт перепиши значения и времена, которые выведет твой ноутбук. `LowerTriangular(A5)` тоже можно использовать как специальное представление нижнетреугольной матрицы, тогда как `tril(A5)` явно создаёт нули выше диагонали.

### 2.4. Линейные модели экономики (4.4.4)

Рассматривается система `(I - A)x = y`, то есть `x = (I - A)⁻¹y`, где компоненты `y` неотрицательны. По критерию из методички матрица продуктивна, если все элементы `(I - A)⁻¹` неотрицательны. Для опровержения продуктивности достаточно найти неотрицательный `y`, для которого решение `x` содержит отрицательную компоненту.

#### 2.4.1. Проверка по определению

In [ ]:
M1 = [1.0 2.0; 3.0 4.0]
I2 = Matrix{Float64}(I, 2, 2)
y_test = [1.0, 0.0]

for scale in (1.0, 1/2, 1/10)
    A_econ = scale .* M1
    C_econ = I2 - A_econ
    Q = C_econ \ I2
    productive = all(Q .>= -1e-10)

    println("\nМасштаб = ", scale)
    println("A = "); display(A_econ)
    println("(I - A)^(-1) = "); display(Q)
    println("Продуктивна: ", productive)

    if !productive
        x_test = Q * y_test
        println("Для y = [1, 0] решение x = ", x_test)
    end
end

#### 2.4.2. Проверка критерием `(I - A)⁻¹ ≥ 0`

Теперь используем матрицу `[[1, 2], [3, 1]]` и её варианты, умноженные на `1/2` и `1/10`.

In [ ]:
M2 = [1.0 2.0; 3.0 1.0]

for scale in (1.0, 1/2, 1/10)
    A_econ = scale .* M2
    Q = (I2 - A_econ) \ I2
    productive = all(Q .>= -1e-10)

    println("\nМасштаб = ", scale)
    println("(I - A)^(-1) = ")
    display(Q)
    println("Все элементы обратной матрицы неотрицательны: ", productive)
end

#### 2.4.3. Проверка спектральным критерием

In [ ]:
for scale in (1.0, 1/2, 1/10)
    A_econ = scale .* M2
    lambda = eigvals(A_econ)
    productive = all(abs.(lambda) .< 1)

    println("\nМасштаб = ", scale)
    println("Собственные значения = ", lambda)
    println("Модули всех собственных значений меньше 1: ", productive)
end

# Матрица из пункта d)
A3 = [0.1 0.2 0.3;
      0.0 0.1 0.2;
      0.0 0.1 0.3]
lambda3 = eigvals(A3)
productive3 = all(abs.(lambda3) .< 1)

println("\nПункт d)")
@show lambda3
@show abs.(lambda3)
@show productive3

## 3. Краткие результаты для вывода

1. **Векторы.** Для `v = [1, 2, 3]` получаем `dot_v = 14`; `outer_v` — матрица `v * v'`.
2. **Системы уравнений.** Смотри классификацию по рангам в ячейках выше. В выводе отчёта отдельно укажи, где решение единственное, где решений бесконечно много и где решений нет.
3. **Диагонализация.** Проверки `V' * A * V ≈ D` и `V * D * V' ≈ A` должны вернуть `true` с учётом погрешности вычислений с плавающей точкой.
4. **Матрица 5×5.** В ноутбуке строятся диагональная матрица собственных значений и нижнетреугольная матрица. Время замеров зависит от компьютера.
5. **Продуктивность в экономической модели.** Для каждого варианта проверь знак элементов `(I - A)⁻¹` и условие `abs(λᵢ) < 1` в спектральной части. Два критерия должны согласовываться там, где они применимы.

## 4. Структура отчёта

1. Титульный лист с номером лабораторной работы и ФИО.
2. Формулировка задания.
3. Описание хода выполнения, исходный код и результаты выполнения; добавь скриншоты ячеек Jupyter.
4. Выводы по результатам.

**Важно:** ячейки подготовлены, но не исполнены в этом файле, потому что в рабочей среде создания ноутбука Julia не установлена. После открытия выполни их в своей Julia/Jupyter и проверь, что вывод совпадает по смыслу с ожидаемыми результатами.